In [13]:
import pandas as pd
import numpy as np

# 1. DATA INGESTION
bowlers_df = pd.read_csv("IPL2025Bowlers.csv")
batters_df = pd.read_csv("IPL2025Batters.csv")


# 2. TRANSFORMATION (Merging & Formatting)
# Use an outer join to ensure we keep pure batters, pure bowlers, and all-rounders
master_roster = pd.merge(bowlers_df, batters_df, on='Player Name', how='outer')

# Rename overlapping columns so they are mathematically precise and readable
column_mapping = {
    'AVG_x': 'Bowling_AVG', 
    'SR_x':  'Bowling_SR',
    'AVG_y': 'Batting_AVG', 
    'SR_y':  'Batting_SR'
}
master_roster.rename(columns=column_mapping, inplace=True)


# 3. DATA CLEANING (Imputation)
# Pure batters/bowlers will have NaNs in their opposite stats. 
# Isolate numeric columns and replace NaNs with 0 to allow for future math.
numeric_columns = master_roster.select_dtypes(include=['number']).columns
master_roster[numeric_columns] = master_roster[numeric_columns].fillna(0)


# 4. VERIFICATION
print(f"✅ Roster Built! Total Players: {master_roster.shape[0]} | Metrics: {master_roster.shape[1]}")
master_roster.head()



# 5. Filtering the data
master_roster_sorted = master_roster.sort_values(by='Runs', ascending=False)
print("sort by the runs in descending orders")
master_roster_sorted.head()


# 6. Filter rows where column 'WKT' > 15
filtered_roster = master_roster[master_roster['WKT'] > 15]
print(filtered_roster)

# 7. display all 
display(master_roster_sorted.head())

# print all the cols name present in the csv
print(master_roster.columns.tolist())





# 1. synthesize a list of fake dates
dates = pd.date_range(start='1/1/2018', end='1/1/2024', periods=len(master_roster))

# 2. Convert them to messy text strings (Object/String type) and add to roster
master_roster['Contract_Date'] = dates.strftime('%m-%d-%Y')

# 3. Verify
print(master_roster[['Player Name', 'Contract_Date']].head())
print("\nCurrent Data Type:", master_roster['Contract_Date'].dtype)


#8. specific pandas function used to convert an entire column of text strings into actual datetime objects
master_roster['Contract_Date'] = pd.to_datetime(master_roster['Contract_Date']) 


# 9. pull just the year out of it and save it into a new column.
master_roster['year'] = master_roster['Contract_Date'].dt.year

# 10. Show the new column
print("\n--- NEW YEAR COLUMN ---")
print(master_roster[['Player Name', 'Contract_Date', 'year']].head())


# 11. The Day 4 Boss Fight: Count players per year
print("\n--- PLAYERS PER YEAR ---")
print(master_roster['year'].value_counts())



# 12. Calculate mean using numpy
mean_runs = np.mean(master_roster['Runs'])
print("Mean of Runs is: ", mean_runs)

# 13. STD(Standard Deviation) of the Runs Coln
std_runs = np.std(master_roster['Runs'])
print("STD of Runs is: ", std_runs)


# 14. Create the new Z-Score column using Vectorized Math
master_roster['Runs_Z_Score'] = (master_roster['Runs'] - mean_runs) / std_runs

# 15. Filter for the Elite Outliers
elite_hitters = master_roster[master_roster['Runs_Z_Score'] > 2.0]

# 16. Display just the name, runs, and their new Z-Score
display(elite_hitters[['Player Name', 'Runs', 'Runs_Z_Score']])

✅ Roster Built! Total Players: 193 | Metrics: 26
sort by the runs in descending orders
             Player Name Team_x   WKT   MAT   INN   OVR   RUNS   BBI  \
16        Arshdeep Singh   PBKS  21.0  17.0  16.0  58.2  518.0  16/3   
25     Bhuvneshwar Kumar    RCB  17.0  14.0  14.0  52.0  483.0  33/3   
45         Harshal Patel    SRH  16.0  13.0  13.0  43.5  430.0  28/4   
53        Jasprit Bumrah     MI  18.0  12.0  12.0  47.2  316.0  22/4   
59        Josh Hazlewood    RCB  22.0  12.0  12.0  44.0  386.0  33/4   
67         Krunal Pandya    RCB  17.0  15.0  15.0  46.0  379.0  45/4   
83          Marco Jansen   PBKS  16.0  14.0  14.0  47.1  434.0  17/3   
94        Mohammed Siraj     GT  16.0  15.0  15.0  57.0  527.0  17/4   
109           Noor Ahmad    CSK  24.0  14.0  14.0  50.0  408.0  18/4   
111          Pat Cummins    SRH  16.0  14.0  14.0  49.4  450.0  19/3   
114      Prasidh Krishna     GT  25.0  15.0  15.0  59.0  488.0  41/4   
141          Sai Kishore     GT  19.0  15.0  15.0

,Player Name,Team_x,WKT,MAT,INN,OVR,RUNS,BBI,Bowling_AVG,ECO,...,Inn,No,HS,Batting_AVG,BF,Batting_SR,100s,50s,4s,6s
142,Sai Sudharsan,NaN,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,...,15.0,1.0,108*,54.21,486.0,156.17,1.0,6.0,88.0,21.0
162,Surya Kumar Yadav,NaN,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,...,16.0,5.0,73*,65.18,427.0,167.91,0.0,5.0,69.0,38.0
179,Virat Kohli,NaN,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,...,15.0,3.0,73*,54.75,454.0,144.71,0.0,8.0,66.0,19.0
158,Shubman Gill,NaN,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,...,15.0,2.0,93*,50.00,417.0,155.87,0.0,6.0,62.0,24.0
89,Mitchell Marsh,NaN,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,...,13.0,0.0,117,48.23,383.0,163.70,1.0,6.0,56.0,37.0


['Player Name', 'Team_x', 'WKT', 'MAT', 'INN', 'OVR', 'RUNS', 'BBI', 'Bowling_AVG', 'ECO', 'Bowling_SR', '4W', '5W', 'Team_y', 'Runs', 'Matches', 'Inn', 'No', 'HS', 'Batting_AVG', 'BF', 'Batting_SR', '100s', '50s', '4s', '6s']
       Player Name Contract_Date
0      Abdul Samad    01-01-2018
1  Abhinav Manohar    01-12-2018
2  Abhishek Sharma    01-23-2018
3    Abishek Porel    02-04-2018
4       Adam Zampa    02-15-2018

Current Data Type: str

--- NEW YEAR COLUMN ---
       Player Name Contract_Date  year
0      Abdul Samad    2018-01-01  2018
1  Abhinav Manohar    2018-01-12  2018
2  Abhishek Sharma    2018-01-23  2018
3    Abishek Porel    2018-02-04  2018
4       Adam Zampa    2018-02-15  2018

--- PLAYERS PER YEAR ---
year
2020    33
2018    32
2019    32
2021    32
2022    32
2023    31
2024     1
Name: count, dtype: int64
Mean of Runs is:  130.39378238341968
STD of Runs is:  173.6533094370496


,Player Name,Runs,Runs_Z_Score
47,Heinrich Klaasen,487.0,2.053553
58,Jos Buttler,538.0,2.347241
61,K L Rahul,539.0,2.353000
89,Mitchell Marsh,627.0,2.859757
106,Nicholas Pooran,524.0,2.266621
113,Prabhsimran Singh,549.0,2.410586
142,Sai Sudharsan,759.0,3.619892
156,Shreyas Iyer,604.0,2.727309
158,Shubman Gill,650.0,2.992205
162,Surya Kumar Yadav,717.0,3.378031
